# Lab 2 · Tracing your Agent — opencode go 版

这是教学视频 **Lab 2: Tracing your Agent** 的本地复刻版：模型从 OpenAI 官方换成 opencode go 订阅的 `mimo-v2.5`，协议统一走 **Chat Completions**（`client.chat.completions.create`），Phoenix tracing 部分与教程一一对应。

| 教程 | 本 Notebook |
|---|---|
| OpenAI 官方 API | opencode go 官方端点 `https://opencode.ai/zen/go/v1` |
| `gpt-4o-mini` | `mimo-v2.5`（官方端点表：`/chat/completions`） |
| `client.chat.completions.create`（Chat Completions） | 同教程，`client.chat.completions.create` |
| `helper.get_openai_api_key()` | macOS Keychain 中的 `opencode-go-api-key` |
| `helper.get_phoenix_endpoint()` | 仓库根目录 `.env.phoenix` 自动发现 |

**协议要点（官方文档 <https://opencode.ai/docs/zh-cn/go/> 的「API 端点」表）：**

Go 网关**按模型分配协议**，不是所有模型都支持 `/responses`：

- `/chat/completions`：`mimo-v2.5`、`ox-alpha-free`、deepseek / kimi / glm 系列、hy3 等（本 Notebook 使用）；
- `/responses`：仅 `grok-4.5`、`gpt-5.6-luna`、`muse-spark-1.2-contributor`；
- `/messages`（Anthropic 协议）：minimax、qwen 系列。

用错协议时网关通常返回**无语义的 `500 Internal server error`**（个别模型返回 `Model ... is not supported for format openai`），排查时先核对官方端点表再探活。API key 存在 macOS Keychain（和 `codex-ox` 同一份），Notebook 运行时动态读取，不落盘、不进 git。

**前置条件：**

- 本地 Phoenix 已启动：`tests/scripts/start-phoenix-local.sh`，浏览器打开 <http://127.0.0.1:6006>
- Kernel 选择 `Python (AI Interviewer · Phoenix Lab)`（即 `ai_interviewer/.venv`，依赖已齐）
- 模型上游可用性会波动，跑主流程前建议先探活确认模型在线（curl POST `/chat/completions` 看是否返回正常 JSON）。


In [1]:
import json
import os
import subprocess
from pathlib import Path

from openai import OpenAI

# Phoenix OTel tracing
from phoenix.otel import register
from openinference.instrumentation.openai import OpenAIInstrumentor

/Users/junjielong/workspace/my_ai_interviewer/ai_interviewer/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 初始化 opencode go 客户端

教程里是 `client = OpenAI(api_key=get_openai_api_key())`；这里只多一样东西：

- `base_url` 指向 opencode go 官方端点。Chat Completions 协议的 `messages` 直接用原生 dict 列表，不再需要教程时代的 message 包装函数。


In [2]:
def get_opencode_go_api_key() -> str:
    """优先读环境变量 OPENCODE_GO_API_KEY；否则从 macOS Keychain 动态读取。"""
    key = os.environ.get("OPENCODE_GO_API_KEY")
    if key:
        return key
    user = os.environ.get("USER", "")
    result = subprocess.run(
        ["security", "find-generic-password", "-a", user, "-s", "opencode-go-api-key", "-w"],
        capture_output=True,
        text=True,
    )
    if result.returncode != 0 or not result.stdout.strip():
        raise RuntimeError(
            "未找到 opencode go API key：请 export OPENCODE_GO_API_KEY=<your key>，"
            "或确认 Keychain 中存在服务名 opencode-go-api-key"
        )
    return result.stdout.strip()


client = OpenAI(
    api_key=get_opencode_go_api_key(),
    base_url="https://opencode.ai/zen/go/v1",  # opencode go 官方端点（Chat Completions 协议）
)

MODEL = "mimo-v2.5"  # 官方端点表：/chat/completions；价格低（$0.14/$0.22 每 1M tokens）、月额度大


## 接入 Phoenix

对应教程的 `phoenix as px` / `register` / `OpenAIInstrumentor` 部分：

1. 先把仓库根目录 `.env.phoenix` 里的本地 Endpoint 和项目名加载进环境变量（`register()` 靠它们发现本地 Phoenix，不上报云端）；
2. `register()` 返回 OTLP TracerProvider；
3. `OpenAIInstrumentor` 给 openai SDK 打补丁——之后每一次 `responses.create()` 都会自动产生带完整属性的 span，业务代码零侵入。

In [3]:
def load_phoenix_env() -> None:
    """加载仓库根目录 .env.phoenix（已存在的环境变量不覆盖）。"""
    for parent in [Path.cwd(), *Path.cwd().parents]:
        candidate = parent / ".env.phoenix"
        if candidate.exists():
            for line in candidate.read_text().splitlines():
                line = line.strip()
                if line and not line.startswith("#") and "=" in line:
                    name, _, value = line.partition("=")
                    os.environ.setdefault(name.strip(), value.strip())
            print(f"已加载配置: {candidate}")
            break


load_phoenix_env()

tracer_provider = register()  # 自动发现 PHOENIX_COLLECTOR_ENDPOINT / PHOENIX_PROJECT_NAME
OpenAIInstrumentor(tracer_provider=tracer_provider).instrument()
print(f"tracing 已开启 -> {os.environ.get('PHOENIX_COLLECTOR_ENDPOINT')}, 项目: {os.environ.get('PHOENIX_PROJECT_NAME')}")

已加载配置: /Users/junjielong/workspace/my_ai_interviewer/.env.phoenix
🔭 OpenTelemetry Tracing Details 🔭
|  Phoenix Project: ai-interviewer-agent-eval
|  Span Processor: SimpleSpanProcessor
|  Collector Endpoint: 127.0.0.1:4317
|  Transport: gRPC
|  Transport Headers: {}
|  
|  Using a default SpanProcessor. `add_span_processor` will overwrite this default.
|  
|  ⚠️ WARNING: It is strongly advised to use a BatchSpanProcessor in production environments.
|  
|  `register` has set this TracerProvider as the global OpenTelemetry default.
|  To disable this behavior, call `register` with `set_global_tracer_provider=False`.

tracing 已开启 -> http://127.0.0.1:6006, 项目: ai-interviewer-agent-eval


## 第 1 次 traced 调用（非流式）

和教程一样，从一次最简单的调用开始。运行后去 Phoenix UI 里应该能看到一条新 Trace：根 span 是 `chat - mimo-v2.5`，展开能看到完整的 prompt/response 文本和 token 用量。


In [4]:
resp = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": "用一句话向后端工程师解释什么是 LLM observability"}],
)
print(resp.choices[0].message.content)
print("\ntoken 用量:", resp.usage)


LLM observability is like adding detailed logging and tracing to your LLM system, letting you see the full chain—from input prompts and internal reasoning steps to output tokens and costs—so you can debug, monitor, and optimize its behavior just like you would with a traditional microservice.

token 用量: CompletionUsage(completion_tokens=214, prompt_tokens=259, total_tokens=473, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=None, audio_tokens=0, reasoning_tokens=0, rejected_prediction_tokens=None, text_tokens=None), prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cache_write_tokens=0, cached_tokens=192, image_tokens=None, text_tokens=None))


## 流式调用

Streaming 下 Instrumentor 会把增量拼回完整文本后再写入 span，所以在 Phoenix 里看到的仍是完整对话。

In [12]:
stream = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": "列出你会考察候选人的两个工程能力，各配一句话理由"}],
    stream=True,
)
for chunk in stream:
    if chunk.choices and chunk.choices[0].delta.content:
        print(chunk.choices[0].delta.content, end="", flush=True)
print()


1. **系统设计与权衡能力** —— 因为工程的核心不是写出能跑的代码，而是在性能、成本、可维护性等真实约束下做出合理的架构取舍。

2. **调试与问题定位能力** —— 因为在生产环境中快速定位并解决故障的水平，往往比从零实现新功能更能反映一个工程师的功底。


## Mini Agent：工具调用循环

教程 Lab 2 的重点是 tracing **agent**。这里实现一个最小 tool-calling 循环：模型决定何时调 `get_interview_stats` 工具 → 本地执行工具并把结果以 `role: "tool"` 消息回传 → 模型基于结果给出最终回答。

Phoenix 侧的预期（OpenAIInstrumentor 只拦截 openai SDK 调用）：

- 两次 `chat.completions.create()` **各自产生一条 Trace**（根 span `chat - mimo-v2.5`）；本地工具执行不产生 span；
- 第二条 Trace 的输入消息里能看到完整历史：user → assistant(tool_calls) → tool(结果)——这正是排查 agent 行为时最常用的视角。

（数据全部虚构，符合本目录的脱敏要求。）


In [5]:
MOCK_DB = {
    "Drake": {"sessions": 12, "avg_score": 82.5, "weak_topics": ["系统设计", "行为面试"]},
}

TOOLS = [
    {
        "type": "function",
        "function": {  # chat completions 的工具声明多包一层 "function"
            "name": "get_interview_stats",
            "description": "查询指定候选人的模拟面试统计",
            "parameters": {
                "type": "object",
                "properties": {"candidate": {"type": "string", "description": "候选人代号"}},
                "required": ["candidate"],
            },
        },
    }
]

from openinference.semconv.trace import OpenInferenceSpanKindValues
from opentelemetry.trace import StatusCode

tracer = tracer_provider.get_tracer(__name__)


def execute_tool(call) -> str:
    """call 是 message.tool_calls 里的条目，模型给的参数 JSON 字符串在 call.function.arguments。"""
    args = json.loads(call.function.arguments)
    with tracer.start_as_current_span("execute_tool", openinference_span_kind=OpenInferenceSpanKindValues.TOOL) as span:
        span.set_input(value = args) 
        span.set_attribute("candidate", args.get("candidate", ""))
        span.set_attribute("tool_name", call.function.name)
        span.set_attribute("tool_id", call.id)
        output = json.dumps(MOCK_DB.get(args.get("candidate", ""), {}), ensure_ascii=False)
        span.set_output(value = output)
        return output


def run_agent(question: str, max_rounds: int = 3) -> str:
    conversation = [{"role": "user", "content": question}]
    with tracer.start_as_current_span("run_agent", openinference_span_kind=OpenInferenceSpanKindValues.AGENT) as span:
        span.set_input(value = question)
        for round_no in range(max_rounds):
            resp = client.chat.completions.create(model=MODEL, messages=conversation, tools=TOOLS)
            msg = resp.choices[0].message
            span.set_output(value = msg.content)
            calls = msg.tool_calls or []
            if not calls:
                return msg.content
            # 把模型的 assistant 消息（含 tool_calls）原样回传，再附上工具结果
            conversation.append(msg.model_dump(exclude_none=True))
            for call in calls:
                conversation.append(
                    {"role": "tool", "tool_call_id": call.id, "content": execute_tool(call)}
                )
            span.set_status(StatusCode.OK)
    return "达到最大轮数仍未得到最终回答"


print(run_agent("查一下 Drake 的模拟面试统计，指出他最需要补强的方向"))


根据您的查询，Drake 的模拟面试统计如下：

**📊 面试概览**
- 总模拟面试次数：12次
- 平均分：82.5分

**🎯 最需要补强的方向**
Drake 在以下两个领域表现相对薄弱，建议优先加强：

1. **系统设计** - 这是技术面试中的核心能力，需要加强架构思维、大规模系统设计以及技术选型方面的训练。
2. **行为面试** - 在展示软技能、团队协作和问题解决经验方面需要更多准备。

**💡 提升建议**
- 针对系统设计：可以练习常见的设计模式（如URL短链、消息队列等），并学习如何清晰地表达设计思路
- 针对行为面试：准备STAR法则（情境-任务-行动-结果）的回答框架，整理自己的项目经历和团队协作案例

这两个方向的提升将显著增强Drake的综合面试表现。需要我为某个具体方向提供更详细的准备建议吗？


## 去 Phoenix 里核对

浏览器打开 <http://127.0.0.1:6006> → 项目 `ai-interviewer-agent-eval`，逐条检查：

1. **根 Trace 数量**：上面每次 `create()` 各一条（非流式、流式、mini agent 两轮）；
2. **Span Attributes**：`llm.input_messages` / `llm.output_messages`（完整 prompt 与回复）、`llm.model_name`、`llm.token_count.prompt/completion/total`；
3. **Agent Trace**：mini agent 的第二条 Trace，输入消息应含 user → assistant(tool_calls) → tool(result) 三层历史；
4. **横向对比**：同一问题多次运行的 latency 和 token 差异——这就是后续固定 Case、做评测基线的入口。


In [6]:
print(
    "Phoenix UI: "
    f"{os.environ.get('PHOENIX_COLLECTOR_ENDPOINT', 'http://127.0.0.1:6006')}"
    f"  (项目: {os.environ.get('PHOENIX_PROJECT_NAME', 'ai-interviewer-agent-eval')})"
)

Phoenix UI: http://127.0.0.1:6006  (项目: ai-interviewer-agent-eval)


---
# Lab 3 扩展 · 组件级评估（Router & Skill Evals）

前面的部分解决的是「**看得见**」（tracing）；这一部分应用 L7 *Lab 3: Adding Router & Skill Evaluations* 的教学点，解决「**测得准**」：不要只看 Agent 最终答得对不对（黑盒），而是把流水线拆开，对 **路由选工具、工具执行、最终表达** 每一环单独量化（白盒）。

| L7 教学点 | 在本 Lab 的落地 |
|---|---|
| 1. `SpanQuery` 从 trace 回收评估数据 | 捞 `LLM` / `TOOL` / `AGENT` 三类 span，组装三份评测数据集 |
| 2. `suppress_tracing()` 防裁判污染 | 所有评估调用包进抑制上下文，附一个可亲眼验证的 A/B 对照实验 |
| 3. LLM-as-a-Judge 结构化裁判 | 给「路由决策」「回答清晰度」打分，产出 label/score/explanation |
| 4. 评估器选型：代码断言 vs LLM 裁判 | 工具返回结构用确定性 JSON 校验（零成本），语义质量才请裁判 |
| 5. 统一打分格式并回写 Phoenix | 结果以 span annotation 写回，UI 里打开任一 span 即可对照 |

> **⚠️ API 迁移提示**：L7 教程基于旧版 phoenix（`px.Client().query_spans` + `llm_classify` + `log_evaluations(SpanEvaluations(...))`）。本环境是 **phoenix 20.x + phoenix-evals 3.x**，这些旧入口已全部移除，对应关系如下：
>
> | L7 教程写法（旧） | 本 Notebook 写法（新） |
> |---|---|
> | `px.Client().query_spans(query)` | `Client().spans.get_spans_dataframe(query=...)` |
> | `llm_classify(df, template, rails, OpenAIModel(...))` | `create_classifier(...) + evaluate_dataframe(df, evaluators=[...])` |
> | 内置模板 `TOOL_CALLING_PROMPT_TEMPLATE` | 新版不内置，自己写 judge prompt（结构更可控） |
> | `log_evaluations([SpanEvaluations(...)])` | `spans.log_span_annotations_dataframe(...)`（span 注解） |

**前置条件**：按顺序跑完上面所有 cell（复用其中的 `client`、`MODEL`、`tracer`、`get_opencode_go_api_key`），本地 Phoenix 在线。

In [7]:
# ── 扩展部分的公共设施 ──────────────────────────────────────────────
import json
import pandas as pd
from datetime import datetime, timezone

from phoenix.client import Client                     # 新版客户端（替代旧 px.Client()）
from phoenix.trace.dsl import SpanQuery               # Span 查询 DSL，用法与教程一致
from openinference.instrumentation import suppress_tracing
from phoenix.evals import LLM, create_classifier, evaluate_dataframe

phoenix_client = Client(base_url=os.environ.get("PHOENIX_COLLECTOR_ENDPOINT", "http://127.0.0.1:6006"))
PROJECT_NAME = os.environ.get("PHOENIX_PROJECT_NAME", "ai-interviewer-agent-eval")

# 裁判模型也走 opencode go 的 Chat Completions（和被测 Agent 同一个便宜渠道）
judge_llm = LLM(
    provider="openai",
    model=MODEL,
    api_key=get_opencode_go_api_key(),
    base_url="https://opencode.ai/zen/go/v1",
)
print("评测基础设施就绪，项目:", PROJECT_NAME)

评测基础设施就绪，项目: ai-interviewer-agent-eval


## Step 0 · 升级 mini Agent：单工具 → 双工具路由器

路由评估要有意义，Agent 必须真的「有的选」。这里给 mini agent 加第二个工具 `recommend_study_plan`，形成 **两个语义相近的工具 + 不调工具** 三类路由决策。

另外两处工程改进（后面查询会用到）：

1. 工具执行 span 的**名字带上具体工具名**：`execute_tool.get_interview_stats` —— 这样能用 `name == '...'` 精确捞某一类工具的执行记录，对应 L7 里 `name == 'generate_visualization'` 的查法；
2. `run_agent_v2` 只在拿到最终回答时才写 span output，且根 span 命名为 `run_agent_v2`，与前面单工具版区分开。

In [8]:
MOCK_STUDY_PLANS = {
    "Drake": {
        "candidate": "Drake",
        "weeks": [
            {"focus": "系统设计", "tasks": ["秒杀系统容量估算", "feed 流架构权衡"]},
            {"focus": "行为面试", "tasks": ["用 STAR 法则整理 3 个项目故事"]},
        ],
    },
}

TOOLS_V2 = [
    {
        "type": "function",
        "function": {
            "name": "get_interview_stats",
            "description": "查询指定候选人的模拟面试统计",
            "parameters": {
                "type": "object",
                "properties": {"candidate": {"type": "string", "description": "候选人代号"}},
                "required": ["candidate"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "recommend_study_plan",
            "description": "根据候选人情况推荐一份补强学习计划",
            "parameters": {
                "type": "object",
                "properties": {"candidate": {"type": "string", "description": "候选人代号"}},
                "required": ["candidate"],
            },
        },
    },
]


def execute_tool_v2(call) -> str:
    """v2：span 名字带上具体工具名，便于后面按 name 精确查询。"""
    args = json.loads(call.function.arguments)
    kind = OpenInferenceSpanKindValues.TOOL
    with tracer.start_as_current_span(f"execute_tool.{call.function.name}", openinference_span_kind=kind) as span:
        span.set_input(value=args)
        span.set_attribute("tool_name", call.function.name)
        span.set_attribute("tool_id", call.id)
        if call.function.name == "get_interview_stats":
            result = MOCK_DB.get(args.get("candidate", ""), {})
        elif call.function.name == "recommend_study_plan":
            result = MOCK_STUDY_PLANS.get(args.get("candidate", ""), {})
        else:
            result = {"error": f"unknown tool: {call.function.name}"}
        output = json.dumps(result, ensure_ascii=False)
        span.set_output(value=output)
        return output


def run_agent_v2(question: str, max_rounds: int = 4) -> str:
    """v2：双工具路由；只在拿到最终回答时写 output；根 span 名为 run_agent_v2。"""
    conversation = [{"role": "user", "content": question}]
    kind = OpenInferenceSpanKindValues.AGENT
    with tracer.start_as_current_span("run_agent_v2", openinference_span_kind=kind) as span:
        span.set_input(value=question)
        for _ in range(max_rounds):
            resp = client.chat.completions.create(model=MODEL, messages=conversation, tools=TOOLS_V2)
            msg = resp.choices[0].message
            calls = msg.tool_calls or []
            if not calls:
                span.set_output(value=msg.content)
                span.set_status(StatusCode.OK)
                return msg.content
            # 把模型的 assistant 消息（含 tool_calls）原样回传，再附上工具结果
            conversation.append(msg.model_dump(exclude_none=True))
            for call in calls:
                conversation.append({"role": "tool", "tool_call_id": call.id, "content": execute_tool_v2(call)})
        span.set_output(value="(达到最大轮数仍未得到最终回答)")
    return "(达到最大轮数仍未得到最终回答)"


print(run_agent_v2("给 Drake 推荐一份补强学习计划")[:200], "...")

## 📚 为 Drake 定制的补强学习计划

根据你的模拟面试数据（共 12 次，平均分 **82.5**），我为你制定了以下两周学习计划：

### 🗓️ 第一周：系统设计
你在这个模块还有提升空间，重点练习：
- **秒杀系统容量估算**：从流量预估、缓存策略、限流降级等角度完整拆解
- **Feed 流架构权衡**：对比推模式 vs 拉模式，理解各自适用场景

### 🗓️ 第二周：行为面 ...


## Step 1 · 固定测试问题集跑批

组件级评估的第一步不是写评估器，而是**准备可复现的被测数据**：固定一组覆盖不同路由决策的问题（该调工具 A 的、该调工具 B 的、不该调任何工具的），批量跑一遍。

跑之前记下时间戳 `RUN_START` —— 后面所有 `SpanQuery` 都用它做时间窗过滤（`start_time=`），保证评测数据集只含本次跑批的 trace，不被历史数据污染。这也是为什么 Step 0 的冒烟调用不会混进评测集：它发生在时间戳之前。

> **⚠️ 实战坑位**：span 从进程导出到 Phoenix 落库有**秒级延迟**，跑完立刻查询会漏掉最后几条 trace（本 Lab 开发时就踩过：6 个问题只查到 5 条）。所以跑批结束后要**轮询等待导出稳定**再进入评估环节。

In [9]:
eval_questions = [
    # 应调 get_interview_stats
    "查一下 Drake 的模拟面试统计",
    "Drake 平均分多少？他最需要补强哪个方向？",
    # 应调 recommend_study_plan
    "给 Drake 推荐一份补强学习计划",
    "帮我看看 Drake 下半年该怎么安排学习",
    # 不应调用任何工具（路由负样本）
    "用一句话向后端工程师解释什么是 token 用量",
    "列出你会考察候选人的两个工程能力，各配一句话理由",
]

import time

RUN_START = datetime.now(timezone.utc)   # 时间窗起点：之后所有查询只认这个时刻之后的 span

for q in eval_questions:
    try:
        answer = run_agent_v2(q)
        print(f"Q: {q}\nA: {answer[:100]}...\n")
    except Exception as e:
        print(f"Q: {q}\n  !! 出错跳过: {e}\n")


def count_spans(where: str) -> int:
    """统计时间窗内满足条件的 span 数量。"""
    df = phoenix_client.spans.get_spans_dataframe(
        query=SpanQuery().where(where),
        project_name=PROJECT_NAME, start_time=RUN_START, timeout=60,
    )
    return len(df)


def wait_for_traces(check, timeout_s: float = 45, interval: float = 4):
    """轮询直到 check() 连续两次结果一致（导出稳定）或超时。

    span 从本地导出到 Phoenix 落库有秒级延迟，查询前必须等它稳定，
    否则评测数据集会随机缺行——这是可观测性系统里典型的最终一致性表现。
    """
    history = []
    deadline = time.time() + timeout_s
    while time.time() < deadline:
        current = check()
        history.append(current)
        if len(history) >= 2 and history[-1] == history[-2]:
            return current
        time.sleep(interval)
    return history[-1]


n_agent = wait_for_traces(lambda: count_spans("span_kind == 'AGENT' and name == 'run_agent_v2'"))
print(f"trace 导出稳定：{n_agent} 条 run_agent_v2 根 trace（预期 {len(eval_questions)}）")

Q: 查一下 Drake 的模拟面试统计
A: 以下是 **Drake** 的模拟面试统计：

| 指标 | 数据 |
|------|------|
| 📊 模拟面试次数 | 12 次 |
| 📈 平均得分 | 82.5 分 |
| ⚠️ 薄弱领...

Q: Drake 平均分多少？他最需要补强哪个方向？
A: 你好！根据Drake的模拟面试数据：

📊 **平均分：82.5分**

🎯 **最需要补强的方向：系统设计和行为面试**

这两个方向确实是技术面试中的关键部分。我看到Drake已经完成了12次模拟...

Q: 给 Drake 推荐一份补强学习计划
A: 根据Drake的模拟面试情况，我为他推荐了一份为期两周的补强学习计划：

## 📚 补强学习计划（Drake）

### **第一周：系统设计强化**
**重点方向**：分布式系统设计与架构思维

*...

Q: 帮我看看 Drake 下半年该怎么安排学习
A: 根据 Drake 的模拟面试数据（已完成 12 次，平均分 82.5），我发现他主要的薄弱环节是 **系统设计** 和 **行为面试**。

基于此，我为他推荐了一份为期两周的集中补强计划：

**第...

Q: 用一句话向后端工程师解释什么是 token 用量
A: Token 用量就像 API 调用中的数据流量，每个 token 代表模型处理的文本片段，用量决定成本和响应速度。...

Q: 列出你会考察候选人的两个工程能力，各配一句话理由
A: # 我会考察的两个工程能力

## 1. **系统设计能力**
> 理由：真实工程问题往往不是"能不能写出来"，而是"能不能在约束条件下设计出可演进、可维护的方案"——这决定了代码是资产还是负债。

...

trace 导出稳定：6 条 run_agent_v2 根 trace（预期 6）


## 教学点 1 · `SpanQuery`：从 Trace 里回收评估数据

评估样本不是手工准备的，而是从真实运行的 trace 里「回收」。三类 span 各有用途：

| span_kind | 对应环节 | 本 Lab 从中提取 |
|---|---|---|
| `LLM` | 每次大模型调用 | 用户问题 + 可用工具列表 + 模型实际选的 tool_calls → **路由数据集** |
| `TOOL` | 每次工具执行 | 工具入参与返回结果 → **代码断言数据集** |
| `AGENT` | Agent 主流程出入口 | 最终回答 → **清晰度裁判数据集** |

三个实用技巧：

1. **只要第一轮调用**：agent 第二轮请求的消息历史里已带 `tool` 角色消息，那是执行后的续问而非路由决策，要滤掉（数 `"role"` 出现次数即可）；
2. **清洗成裁判易读的格式**：`input.value` 是整个请求体的 JSON 串，直接喂给裁判噪声很大，先解析出纯 user 问题；
3. **模型的实际选择在响应里**：从 `output.value` 解析出 `tool_calls`，「有哪些工具可选」（`llm.tools`）和「实际选了什么」要分开两列。

In [10]:
def extract_user_question(raw: str) -> str:
    """从请求体 JSON 中提取 user 提问，裁判只需要看这个。"""
    try:
        msgs = json.loads(raw).get("messages", [])
        return "\n".join(m["content"] for m in msgs if m.get("role") == "user")
    except Exception:
        return str(raw)[:500]


def extract_tool_calls(raw: str) -> str:
    """从响应体 JSON 中提取模型实际的 tool_calls 决策。"""
    try:
        msg = json.loads(raw)["choices"][0]["message"]
        calls = [
            {"name": c["function"]["name"], "arguments": c["function"]["arguments"]}
            for c in (msg.get("tool_calls") or [])
        ]
        return json.dumps(calls, ensure_ascii=False)
    except Exception:
        return "[]"


router_query = SpanQuery().where("span_kind == 'LLM'").select(
    raw_request="input.value",      # 完整请求体（含 messages）
    available_tools="llm.tools",    # 本次调用暴露给模型的全部工具
    raw_response="output.value",    # 完整响应体（含 tool_calls）
)
router_df = phoenix_client.spans.get_spans_dataframe(
    query=router_query, project_name=PROJECT_NAME, start_time=RUN_START, timeout=60,
)

is_first_round = router_df["raw_request"].fillna("").map(lambda s: s.count('"role"') <= 1)
router_df = (
    router_df[is_first_round]                                            # 只要第一轮的路由决策
    .dropna(subset=["available_tools"])                                  # 只要传了 tools 的调用
    .assign(                                                             # 清洗出裁判要看的三个字段
        question=lambda d: d["raw_request"].map(extract_user_question),
        chosen_tool_call=lambda d: d["raw_response"].map(extract_tool_calls),
        tool_definitions=lambda d: d["available_tools"].map(lambda t: json.dumps(t, ensure_ascii=False)),
    )[["question", "chosen_tool_call", "tool_definitions"]]
)
print(f"路由评估数据集：{len(router_df)} 行（预期 ≈ {len(eval_questions)} 行，每题一条第一轮调用）")
router_df.head()

路由评估数据集：6 行（预期 ≈ 6 行，每题一条第一轮调用）


,question,chosen_tool_call,tool_definitions
context.span_id,,,
5644e896a9680c7f,列出你会考察候选人的两个工程能力，各配一句话理由,[],"[{""tool"": {""json_schema"": ""{\""type\"": \""functi..."
b68a6e815551ecc2,用一句话向后端工程师解释什么是 token 用量,[],"[{""tool"": {""json_schema"": ""{\""type\"": \""functi..."
16dde0a14bc35ca9,帮我看看 Drake 下半年该怎么安排学习,"[{""name"": ""get_interview_stats"", ""arguments"": ...","[{""tool"": {""json_schema"": ""{\""type\"": \""functi..."
1cd0bbfe71a86ab9,给 Drake 推荐一份补强学习计划,"[{""name"": ""recommend_study_plan"", ""arguments"":...","[{""tool"": {""json_schema"": ""{\""type\"": \""functi..."
f2a20d6524dc563d,Drake 平均分多少？他最需要补强哪个方向？,"[{""name"": ""get_interview_stats"", ""arguments"": ...","[{""tool"": {""json_schema"": ""{\""type\"": \""functi..."


## 教学点 2 · `suppress_tracing()`：别让裁判污染被测数据

评估器本身也是 LLM 调用，而 openai SDK 已被全局 instrumentation 包住——**不处理的话，裁判自己的 prompt 和回答也会上报成新的 LLM span**，混进同一个项目。

后果是**循环污染**：下次再用 `SpanQuery` 捞 LLM spans 时，会把裁判的数据当成被测样本，评测集越滚越脏，而且全是同一分布的伪数据。解法是把所有评估调用包进 `suppress_tracing():` 上下文。

下面先做个 A/B 实验亲眼验证：两个探针问题各问一次裁判，一次不抑制、一次抑制；考虑到导出延迟，**等 trace 稳定后一次性统计**两个标记各自出现的次数（开发本 Lab 时先写了"问完立刻查"的版本，结果因为导出延迟得到完全相反的结论——测量本身也要考虑可观测性系统的最终一致性）。

In [11]:
SUPPRESS_MARKER_A = "[AB_DEMO_UNSAFE]"
SUPPRESS_MARKER_B = "[AB_DEMO_SAFE]"

demo_clf = create_classifier(
    name="suppress_demo",
    prompt_template=(
        "判断回答是否清晰。\n[BEGIN DATA]\n问题: {question}\n回答: {response}\n[END DATA]\n"
        "先简短解释，再给出标签 clear 或 unclear"
    ),
    llm=judge_llm,
    choices=["clear", "unclear"],   # 不带分数映射的写法：只有 label，score 为 None
)


def llm_span_count() -> int:
    return count_spans("span_kind == 'LLM'")


probe_a = {"question": f"{SUPPRESS_MARKER_A} 什么是 RAG？", "response": "RAG 是先检索外部知识再生成回答的技术。"}
probe_b = {"question": f"{SUPPRESS_MARKER_B} 什么是 RAG？", "response": "RAG 就是模型自己想答案，跟外部资料没关系。"}

demo_clf.evaluate(probe_a)                      # A：故意不抑制
with suppress_tracing():                        # B：抑制
    demo_clf.evaluate(probe_b)

wait_for_traces(llm_span_count)                 # 等导出稳定再统计，避免竞态

trace_df = phoenix_client.spans.get_spans_dataframe(
    query=SpanQuery().where("span_kind == 'LLM'").select(text="input.value"),
    project_name=PROJECT_NAME, start_time=RUN_START, timeout=60,
)
texts = trace_df["text"].fillna("")
count_a = int(texts.str.contains(SUPPRESS_MARKER_A, regex=False).sum())
count_b = int(texts.str.contains(SUPPRESS_MARKER_B, regex=False).sum())

print(f"未抑制的裁判调用出现在 trace 里: {count_a} 次（预期 >= 1）")
print(f"抑制后   的裁判调用出现在 trace 里: {count_b} 次（预期 0）")

未抑制的裁判调用出现在 trace 里: 2 次（预期 >= 1）
抑制后   的裁判调用出现在 trace 里: 0 次（预期 0）


## 教学点 3 · LLM-as-a-Judge：给路由决策打分

新版 API 的裁判三件套（对应教程里的 `llm_classify`）：

| 新 API | 作用 | 对应旧版概念 |
|---|---|---|
| `create_classifier(name, prompt_template, llm, choices)` | 定义裁判：模板占位符 ↔ 数据列名自动对应 | `template` + `rails` |
| `choices={"correct": 1, "incorrect": 0}` | 标签→分数映射（用 dict 才会产出 score，list 则只有 label） | 手动 `map` 补 score |
| `evaluate_dataframe(df, evaluators=[clf])` | 逐行跑裁判，新增 `{name}_score` 列（dict 含 label/score/explanation） | 返回值结构不同 |

两个注意点：

- 模板占位符名必须和数据集列名一致（`{question}` ↔ `question` 列）；
- 分类器内部走**结构化输出/工具调用**拿结果，所以要求裁判模型支持 function calling——`mimo-v2.5` 满足。

**如何解读结果**：如果正确率偏低，先读 `explanation` 列再下结论。可能真的是路由选错了，也可能是裁判规则太严——比如 Agent 对「推荐学习计划」类问题先调 `get_interview_stats` 拿数据、下一轮再给建议，这种合理的多轮策略会被「单轮决策」视角判为 incorrect。这正是组件级评估的价值：错误被定位到具体环节、且附带可读的原因，改 prompt 规则还是改工具描述，依据就在眼前。

In [12]:
ROUTER_JUDGE_PROMPT = """你是一名严格的 Agent 评审员。给定用户问题、可选工具清单、以及 Agent 实际发起的工具调用（可能为空 []），判断这次路由决策是否正确。

规则：
- 问题需要事实数据或具体操作时，应该调用合适的工具；
- 问题只是常识性或概念性问答时，不应该调用任何工具（tool call 为空 [] 才算正确）;
- 选错工具或编造不存在的工具都算 incorrect。

[BEGIN DATA]
************
[Question]: {question}
[Tool Definitions]: {tool_definitions}
[Chosen Tool Call]: {chosen_tool_call}
************
[END DATA]

EXPLANATION: 先逐步分析问题需要什么信息、每个工具能提供什么，再得出结论（不要在解释开头就给标签）。
LABEL: 只写 correct 或 incorrect"""

router_clf = create_classifier(
    name="router_correctness",
    prompt_template=ROUTER_JUDGE_PROMPT,
    llm=judge_llm,
    choices={"correct": 1, "incorrect": 0},   # 标签→分数映射
)

span_ids = router_df.index.copy()             # 防御：确认批量评估不丢索引
with suppress_tracing():                      # 教学点 2：评估全程不上报
    router_eval_df = evaluate_dataframe(router_df, evaluators=[router_clf])
if list(router_eval_df.index) != list(span_ids):
    router_eval_df.index = span_ids

# 展平成统一的 label / score / explanation 三列（教学点 5 复用这套结构）
router_eval_df["label"] = router_eval_df["router_correctness_score"].map(lambda s: s["label"])
router_eval_df["score"] = router_eval_df["router_correctness_score"].map(lambda s: s["score"])
router_eval_df["explanation"] = router_eval_df["router_correctness_score"].map(lambda s: s.get("explanation", ""))

print(f"路由正确率: {router_eval_df['score'].mean():.0%}  ({int(router_eval_df['score'].sum())}/{len(router_eval_df)})")
router_eval_df[["question", "chosen_tool_call", "label", "score"]]

/var/folders/n_/5znndzt93mjcksm3cnmw_kgc0000gn/T/ipykernel_24594/2512105164.py:28: DeprecationWarning: Positional arguments for evaluate_dataframe are deprecated and will be removed in a future version. Please use keyword arguments instead.
  router_eval_df = evaluate_dataframe(router_df, evaluators=[router_clf])
Evaluating Dataframe |██████████| 6/6 (100.0%) | ⏳ 00:42<00:00 |  7.07s/it

路由正确率: 100%  (6/6)


,question,chosen_tool_call,label,score
context.span_id,,,,
5644e896a9680c7f,列出你会考察候选人的两个工程能力，各配一句话理由,[],correct,1
b68a6e815551ecc2,用一句话向后端工程师解释什么是 token 用量,[],correct,1
16dde0a14bc35ca9,帮我看看 Drake 下半年该怎么安排学习,"[{""name"": ""get_interview_stats"", ""arguments"": ...",correct,1
1cd0bbfe71a86ab9,给 Drake 推荐一份补强学习计划,"[{""name"": ""recommend_study_plan"", ""arguments"":...",correct,1
f2a20d6524dc563d,Drake 平均分多少？他最需要补强哪个方向？,"[{""name"": ""get_interview_stats"", ""arguments"": ...",correct,1
4644942e36f655ac,查一下 Drake 的模拟面试统计,"[{""name"": ""get_interview_stats"", ""arguments"": ...",correct,1


## 教学点 4 · 评估器选型：代码断言 vs LLM 裁判

不是什么都要请 LLM 当裁判。**能用确定性代码验证的属性，就用代码评估器**——零成本、100% 可靠、完全可复现。L7 用 `exec()` 验证生成的 Python 代码可不可运行；本 Lab 的等价物是：校验 `get_interview_stats` 的返回是否为合法 JSON 且字段类型符合约定。

| 被评属性 | 性质 | 手段 | 成本 |
|---|---|---|---|
| 工具返回结构合法（下面这个） | 客观、机械可验 | **代码断言** | ≈ 0 |
| 路由决策正确（教学点 3） | 半客观（需理解意图） | LLM 裁判 | 每次 1 次调用 |
| 回答清晰度（下一节） | 主观（语义质量） | LLM 裁判 | 每次 1 次调用 |

经验法则：**先问「这个属性能不能写成断言」，不能才上裁判**。

In [13]:
tool_query = SpanQuery().where(
    "span_kind == 'TOOL' and name == 'execute_tool.get_interview_stats'"
).select(
    tool_input="input.value",
    tool_output="output.value",
)
tool_df = phoenix_client.spans.get_spans_dataframe(
    query=tool_query, project_name=PROJECT_NAME, start_time=RUN_START, timeout=60,
)
print(f"{len(tool_df)} 次 get_interview_stats 工具执行")


def stats_output_is_valid(output) -> bool:
    """确定性断言：合法 JSON + 约定字段类型正确。"""
    try:
        data = json.loads(output)
        return (
            isinstance(data.get("sessions"), int)
            and isinstance(data.get("avg_score"), (int, float))
            and isinstance(data.get("weak_topics"), list)
        )
    except Exception:
        return False


tool_eval_df = tool_df.copy()
validity = tool_eval_df["tool_output"].map(stats_output_is_valid)
tool_eval_df["label"] = validity.map({True: "valid", False: "invalid"})
tool_eval_df["score"] = validity.astype(int)          # True/False -> 1/0，与裁判输出同构
tool_eval_df["explanation"] = tool_eval_df["tool_output"].map(
    lambda o: "schema ok" if stats_output_is_valid(o) else f"bad output: {str(o)[:80]}"
)

print(f"工具输出有效率: {tool_eval_df['score'].mean():.0%}")
tool_eval_df[["tool_input", "label", "score"]]

3 次 get_interview_stats 工具执行
工具输出有效率: 100%


,tool_input,label,score
context.span_id,,,
0f22fd1be6de3e83,"{""candidate"": ""Drake""}",valid,1
b559bf2538fb371e,"{""candidate"": ""Drake""}",valid,1
7c525fb5d37c9590,"{""candidate"": ""Drake""}",valid,1


## 再来一个 LLM 裁判 · 回答清晰度（对应 Tool 2 分析质量）

最后评估整体表达层：从 `AGENT` 根 span 捞最终回答，让裁判按 clear / unclear 打分。这是典型的主观维度——只能靠语义裁判，写不出确定性规则。

prompt 结构值得抄走：**定义清楚每一档的含义 → [BEGIN DATA] 塞样本 → 要求先解释后单独给标签**（避免裁判偷懒直接甩标签）。

In [14]:
CLARITY_JUDGE_PROMPT = """你将看到一个问题和一个回答。请评估回答的清晰度：清晰的回答精确、连贯、直击问题，不引入不必要的复杂或歧义；模糊的回答含糊、松散或难以理解（即使事实正确也算 unclear）。

[BEGIN DATA]
************
[Question]: {question}
[Response]: {response}
************
[END DATA]

EXPLANATION: <你的逐步分析>
LABEL: "clear" 或 "unclear" 二选一"""

agent_query = SpanQuery().where("span_kind == 'AGENT' and name == 'run_agent_v2'").select(
    question="input.value",
    response="output.value",
)
clarity_df = phoenix_client.spans.get_spans_dataframe(
    query=agent_query, project_name=PROJECT_NAME, start_time=RUN_START, timeout=60,
).dropna(subset=["response"])

clarity_clf = create_classifier(
    name="response_clarity",
    prompt_template=CLARITY_JUDGE_PROMPT,
    llm=judge_llm,
    choices={"clear": 1, "unclear": 0},
)

with suppress_tracing():
    clarity_eval_df = evaluate_dataframe(clarity_df, evaluators=[clarity_clf])

clarity_eval_df["label"] = clarity_eval_df["response_clarity_score"].map(lambda s: s["label"])
clarity_eval_df["score"] = clarity_eval_df["response_clarity_score"].map(lambda s: s["score"])
clarity_eval_df["explanation"] = clarity_eval_df["response_clarity_score"].map(lambda s: s.get("explanation", ""))

print(f"回答清晰率: {clarity_eval_df['score'].mean():.0%}  ({int(clarity_eval_df['score'].sum())}/{len(clarity_eval_df)})")
clarity_eval_df[["question", "label", "score"]]

/var/folders/n_/5znndzt93mjcksm3cnmw_kgc0000gn/T/ipykernel_24594/3507444765.py:29: DeprecationWarning: Positional arguments for evaluate_dataframe are deprecated and will be removed in a future version. Please use keyword arguments instead.
  clarity_eval_df = evaluate_dataframe(clarity_df, evaluators=[clarity_clf])
Evaluating Dataframe |██████████| 6/6 (100.0%) | ⏳ 00:40<00:00 |  6.76s/it

回答清晰率: 100%  (6/6)


,question,label,score
context.span_id,,,
85ed10b035e59281,列出你会考察候选人的两个工程能力，各配一句话理由,clear,1
9a193e53442c2c48,用一句话向后端工程师解释什么是 token 用量,clear,1
980992e1fd7631b2,帮我看看 Drake 下半年该怎么安排学习,clear,1
b3be4985173797d1,给 Drake 推荐一份补强学习计划,clear,1
2831d5b47a4fe070,Drake 平均分多少？他最需要补强哪个方向？,clear,1
ad523ca202ae81ca,查一下 Drake 的模拟面试统计,clear,1


## 教学点 5 · 统一格式，回写 Phoenix

三套评估器的产出统一成了同样的结构：`label`（人读标签）+ `score`（0/1 数值）+ `explanation`（理由）。统一之后才能跨评估器聚合出「**哪个环节最薄弱**」这类指标。

回写在新版里变成了 **span annotation**：每条评估结果挂到对应 span_id 上。UI 里打开任一 span，Annotations 区就能看到裁判给的标签、分数和解释——这正是排查「某次为什么被判错」的入口。（旧版的 `SpanEvaluations` + `log_evaluations` 已废弃。）

In [15]:
def to_annotation_df(eval_df: pd.DataFrame) -> pd.DataFrame:
    """index 即 context.span_id，附统一三列。"""
    out = eval_df[["label", "score", "explanation"]].copy()
    out.index.name = "context.span_id"
    return out


for name, df_, kind in [
    ("router_correctness", router_eval_df, "LLM"),     # 裁判产出的 -> LLM
    ("tool_output_validity", tool_eval_df, "CODE"),    # 代码断言产出的 -> CODE
    ("response_clarity", clarity_eval_df, "LLM"),
]:
    res = phoenix_client.spans.log_span_annotations_dataframe(
        dataframe=to_annotation_df(df_), annotation_name=name, annotator_kind=kind, sync=True,
    )
    print(f"已回写 {len(res)} 条 -> {name} ({kind})")

print()
print("组件健康度汇总:")
print(f"  路由正确率   : {router_eval_df['score'].mean():.0%}")
print(f"  工具输出有效率: {tool_eval_df['score'].mean():.0%}")
print(f"  回答清晰率   : {clarity_eval_df['score'].mean():.0%}")

已回写 6 条 -> router_correctness (LLM)
已回写 3 条 -> tool_output_validity (CODE)
已回写 6 条 -> response_clarity (LLM)

组件健康度汇总:
  路由正确率   : 100%
  工具输出有效率: 100%
  回答清晰率   : 100%


## 去 Phoenix 里核对

浏览器打开 <http://127.0.0.1:6006> → 项目 `ai-interviewer-agent-eval`，逐条检查：

1. **Trace 数量**：Step 1 跑批产生 6 条 `run_agent_v2` 根 trace；A/B 实验中只有未抑制的那次裁判调用进了 trace；
2. **Span 形态**：每次 agent 运行 = 1 个 `AGENT` span + N 个 `LLM` span + M 个 `execute_tool.*` TOOL span，树形嵌套一目了然；
3. **Annotations**：点开第一轮 `chat - mimo-v2.5` span（应有 `router_correctness`）、`execute_tool.get_interview_stats` span（应有 `tool_output_validity`）、`run_agent_v2` span（应有 `response_clarity`），都能看到 label / score / explanation；
4. **横向对比**：上面打印的三项均值就是组件级健康度——哪个环节分低修哪个。

## 方法论总结：组件级评估五步工作流

```
┌─────────────┐   ┌─────────────┐   ┌───────────────┐   ┌───────────┐   ┌───────────────┐
│ 1 固定问题集 │ → │ 2 记录时间戳 │ → │ 3 SpanQuery    │ → │ 4 分环评估 │ → │ 5 统一格式回写 │
│ 覆盖各类路由 │   │ 圈定数据窗口 │   │ 回收三类 span  │   │ 断言+裁判  │   │ annotation    │
└─────────────┘   └─────────────┘   └───────────────┘   └───────────┘   └───────────────┘
```

> 黑盒评估只能告诉你「错了」；组件级评估才能告诉你**错在哪一环**——路由选错？工具返回坏了？还是数据对了但讲不清楚？
>
> 下一步的自然延伸：把这组固定问题和期望标签沉淀成 golden dataset，用 Phoenix 的 Experiments 做回归评测（改 prompt / 换模型后一键重跑对比）。